# Comparaison des 4 pipelines
Agrège les fichiers `results/*.json` produits par les notebooks 01 à 04 (à déposer dans le dossier `results/` du dépôt) et génère les graphiques du rapport dans `figures/`. Ce notebook ne fait aucune inférence : il peut tourner sur n'importe quel environnement.

In [ ]:
# Récupération du dépôt : code commun (données, modèle, protocole de mesure)
import os, subprocess, sys
REPO = "https://github.com/Romain-LM-Dev/TP1_deploiement_IA.git"
if os.path.exists("../common.py"):          # exécution locale depuis notebooks/
    os.chdir("..")
elif not os.path.exists("common.py"):       # Google Colab
    if not os.path.exists("TP1_deploiement_IA"):
        subprocess.run(["git", "clone", "-q", REPO], check=True)
    os.chdir("TP1_deploiement_IA")
sys.path.insert(0, os.getcwd())
import common as C
C.env_info()

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
results, metas = C.load_results()
display(pd.DataFrame({k: m["env"] for k, m in metas.items()}))

## 1. Débit, temps d'inférence, précision et énergie

In [ ]:
df = C.plot_summary(results, "Comparaison des 4 pipelines", "figures/05_comparaison.png")
display(df[C.TABLE_COLS].round(4))
os.makedirs("results", exist_ok=True)
df[C.TABLE_COLS].to_csv("results/synthese.csv", index=False)

## 2. Accélération par rapport à PyTorch eager
Chaque notebook a mesuré sa propre référence *eager* sur la même machine et dans la même session. Le rapport débit / débit eager neutralise donc les écarts de matériel entre sessions Colab.

In [ ]:
df["speedup"] = df.throughput_mean / df.groupby("pipeline").throughput_mean.transform("first")
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(df.pipeline + "\n" + df.variant, df.speedup, color=["#4C72B0" if d == "cpu" else "#55A868" for d in df.device])
ax.axhline(1, color="grey", lw=.8)
for i, v in enumerate(df.speedup):
    ax.text(i, v, f"×{v:.2f}", ha="center", va="bottom")
ax.set_ylabel("Débit / débit eager")
ax.set_title("Accélération par rapport à PyTorch eager (même session)")
ax.tick_params(axis="x", labelrotation=45, labelsize=8)
fig.tight_layout(); fig.savefig("figures/05_speedup.png", dpi=130); plt.show()

## 3. Cohérence des prédictions entre pipelines
Taux d'accord des classes prédites (Top-1 ImageNet, plus strict que les 10 classes CIFAR) entre toutes les variantes. 100 % partout = les optimisations ne changent aucune prédiction.

In [ ]:
labels = [f"{r['pipeline']} | {r['variant']}" for r in results]
P = np.array([r["imagenet_top1"] for r in results])
A = (P[:, None, :] == P[None, :, :]).mean(-1)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(A, vmin=min(.9, A.min()), vmax=1, cmap="viridis")
ax.set_xticks(range(len(labels)), labels, rotation=90, fontsize=8)
ax.set_yticks(range(len(labels)), labels, fontsize=8)
for i in range(len(A)):
    for j in range(len(A)):
        ax.text(j, i, f"{A[i, j]:.0%}", ha="center", va="center", fontsize=7, color="w" if A[i, j] < .97 else "k")
fig.colorbar(im); ax.set_title("Accord des prédictions Top-1 ImageNet"); fig.tight_layout()
fig.savefig("figures/05_accord_predictions.png", dpi=130); plt.show()
cifar = np.array([r["cifar_pred"] for r in results])
print("Images dont la prédiction CIFAR diffère selon la variante :", np.where((cifar != cifar[0]).any(0))[0].tolist())

## 4. Détail par image (prédiction de la référence vs vérité terrain)

In [ ]:
y = np.array(C.load_data()[1])
pd.DataFrame({"vérité": [C.CIFAR_CLASSES[i] for i in y],
              "prédiction": [C.CIFAR_CLASSES[i] for i in cifar[0]],
              "correct": cifar[0] == y}).groupby("vérité").correct.agg(["sum", "count"])